# Analisis Kesesuaian Star Rating dan Sentimen Ulasan
## Visualisasi Data Mendalam & Perbandingan Multi-Model

Notebook ini dirancang untuk memberikan analisis komprehensif mulai dari pemahaman data (EDA) hingga evaluasi model mendalam.

### 1. Inisialisasi dan Pemuatan Data

In [ ]:
import pandas as pd
import numpy as np
import re
import kagglehub
from kagglehub import KaggleDatasetAdapter
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
from collections import Counter

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix, 
    balanced_accuracy_score, f1_score, roc_curve, auc, precision_recall_curve
)
from imblearn.over_sampling import RandomOverSampler

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('stopwords')
nltk.download('wordnet')

sns.set_theme(style="whitegrid", palette="pastel")
plt.rcParams['figure.figsize'] = (12, 8)

In [ ]:
df = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "hassaanmustafavi/google-play-app-reviews-dataset",
  "GooglePlay_App_Data.csv",
)
df = df.dropna(subset=['review_description']).rename(columns={"review_description": "review", "rating": "star_rating"})

df['sentiment'] = df['star_rating'].apply(lambda x:
    'positive' if x >= 4 else 'neutral' if x == 3 else 'negative'
)
df['review_len'] = df['review'].apply(len)
df['word_count'] = df['review'].apply(lambda x: len(str(x).split()))

### 2. Exploratory Data Analysis (EDA)
Memahami karakteristik dataset melalui berbagai visualisasi.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(16, 6))

# 1. Distribusi Star Rating
sns.countplot(x='star_rating', data=df, ax=ax[0], palette='viridis')
ax[0].set_title('Distribusi Star Rating (1-5)')

# 2. Distribusi Sentimen
df['sentiment'].value_counts().plot.pie(autopct='%1.1f%%', ax=ax[1], explode=[0.05]*3, shadow=True)
ax[1].set_title('Proporsi Sentimen')
plt.show()

In [ ]:
# Distribusi Panjang Ulasan per Sentimen
plt.figure(figsize=(12, 6))
sns.boxplot(x='sentiment', y='review_len', data=df, showfliers=False, palette='Set2')
plt.title('Panjang Karakter Ulasan per Kategori Sentimen')
plt.ylabel('Panjang Karakter (Tanpa Outliers)')
plt.show()

### 3. Preprocessing & N-Gram Analysis
Melihat kata-kata yang paling sering muncul sebelum masuk ke pemodelan.

In [ ]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = re.sub(r'[^a-z\s]', '', text.lower())
    return " ".join([lemmatizer.lemmatize(w) for w in text.split() if w not in stop_words])

df['clean_review'] = df['review'].apply(clean_text)

In [ ]:
def get_top_ngram(corpus, n=None, top_n=10):
    vec = TfidfVectorizer(ngram_range=(n, n)).fit(corpus)
    bag_of_words = vec.transform(corpus)
    sum_words = bag_of_words.sum(axis=0)
    words_freq = [(word, sum_words[0, idx]) for word, idx in vec.vocabulary_.items()]
    words_freq = sorted(words_freq, key=lambda x: x[1], reverse=True)
    return words_freq[:top_n]

top_bigrams = get_top_ngram(df['clean_review'], n=2, top_n=15)
x_bigram, y_bigram = map(list, zip(*top_bigrams))

plt.figure(figsize=(12, 6))
sns.barplot(x=y_bigram, y=x_bigram, palette='rocket')
plt.title('Top 15 Bigrams pada Ulasan (TF-IDF Weight)')
plt.show()

### 4. Perbandingan Model & Visualisasi Evaluasi

In [ ]:
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X = vectorizer.fit_transform(df['clean_review'])
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
ros = RandomOverSampler(random_state=42)
X_train_res, y_train_res = ros.fit_resample(X_train, y_train)

models = {
    "Naive Bayes": MultinomialNB(),
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Linear SVM": LinearSVC(dual=False)
}

all_preds = {}
for name, model in models.items():
    model.fit(X_train_res, y_train_res)
    all_preds[name] = model.predict(X_test)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 5))
for i, (name, pred) in enumerate(all_preds.items()):
    cm = confusion_matrix(y_test, pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i], 
                xticklabels=['neg', 'neu', 'pos'], yticklabels=['neg', 'neu', 'pos'])
    axes[i].set_title(f'Confusion Matrix: {name}')
plt.tight_layout()
plt.show()

### 5. Analisis Kesesuaian (Mismatch Analysis)
Visualisasi untuk menjawab rumusan masalah utama skripsi.

In [ ]:
best_model_name = "Logistic Regression"
y_pred = all_preds[best_model_name]

mismatch_df = df.iloc[y_test.index].copy()
mismatch_df['pred'] = y_pred
mismatch_df['is_correct'] = mismatch_df['sentiment'] == mismatch_df['pred']

accuracy_per_rating = mismatch_df.groupby('star_rating')['is_correct'].mean()
plt.figure(figsize=(10, 5))
sns.lineplot(x=accuracy_per_rating.index, y=accuracy_per_rating.values, marker='o', color='red', linewidth=3)
plt.title('Tingkat Kesesuaian (Akurasi) per Star Rating')
plt.ylabel('Kesesuaian (0.0 - 1.0)')
plt.xlabel('Star Rating')
plt.ylim(0, 1.1)
plt.show()

### 6. Export for Tableau

In [ ]:
mismatch_df.to_csv("tableau_full_analysis.csv", index=False)
print("Data lengkap telah diekspor untuk Tableau!")